# Export Normalized Rotational Drag Plot as SVG
Reproduces the normalized scatter plot from trial7.ipynb and saves it as an SVG file.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import ndimage

In [2]:
# Standard viscosity mapping
viscosity_mapping = {
    '350cp': 412.07,
    '1kcp': 1073.33,
    '2kcp': 3345.33,
    '4kcp': 6603.00,
    '5kcp': 5861.33,
    '8kcp': 8946.67,
    '10kcp': 9152.00,
    '12.5kcp': 14576.67,
    '15kcp': 19036.67,
    '20kcp': 24396.67,
    '25kcp': 22760.00,
    '30kcp': 31903.33,
    '35kcp': 63253.33,
    '40kcp': 62756.67,
    '45kcp': 40820.00,
    '50kcp': 79653.33,
    '55kcp': 48553.33,
    '60kcp': 68953.33,
    '70kcp': 87046.14,
    '75kcp': 70730.00,
    '80kcp': 103800.00,
    '90kcp': 102466.67,
    '95kcp': 93400.00,
    '100kcp': 124033.33
}

def extract_viscosity(viscosity_sample):
    viscosity_label_str = viscosity_sample.replace('_t/rpm', '')
    if viscosity_label_str in viscosity_mapping:
        return viscosity_mapping[viscosity_label_str]
    if 'kcp' in viscosity_label_str:
        return float(viscosity_label_str.replace('kcp', '')) * 1000
    else:
        return float(viscosity_label_str.replace('cp', ''))

In [3]:
# Load and process data
pathfile_data = 'Data/Manual_Auto/timing_v2.csv'
df_raw = pd.read_csv(pathfile_data)

torque_columns = df_raw.columns[2:]
for col in torque_columns:
    viscosity_label = col.split('_')[0]
    rpm_value = float(col.split('_rpm_')[-1])
    new_col_name = f"{viscosity_label}_t/rpm"
    df_raw[new_col_name] = df_raw[col] / rpm_value

df_raw = df_raw.drop(columns=torque_columns)
df = df_raw.rename(columns={'Z-Height': 'h_mm'})
trpm_columns = [col for col in df.columns if 't/rpm' in col]

# Filter to Elapsed_Time_s > 90 and compute mean per height
df_filtered = df[df['Elapsed_Time_s'] > 90].copy()
df_new = df_filtered.groupby('h_mm')[trpm_columns].mean().reset_index()
print(f"df_new shape: {df_new.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'Data/Manual_Auto/timing_v2.csv'

In [ ]:
# Transition point detection
def find_transition_point(height_data, trpm_data, smoothing_window=5):
    combined_data = sorted(zip(height_data, trpm_data), key=lambda x: x[0], reverse=True)
    heights = np.array([p[0] for p in combined_data])
    trpm_values = np.array([p[1] for p in combined_data])

    valid_mask = ~(np.isnan(heights) | np.isnan(trpm_values))
    heights = heights[valid_mask]
    trpm_values = trpm_values[valid_mask]

    if len(heights) < smoothing_window * 2:
        return None

    height_75th = np.percentile(heights, 75)
    mask = heights <= height_75th
    heights_f = heights[mask]
    trpm_f = trpm_values[mask]

    if len(heights_f) < smoothing_window * 2:
        return None

    trpm_smoothed = ndimage.uniform_filter1d(trpm_f, size=smoothing_window)
    first_deriv = np.gradient(trpm_smoothed, heights_f)
    second_deriv = np.gradient(first_deriv, heights_f)

    for i in range(1, len(second_deriv)):
        if second_deriv[i-1] > 0 and second_deriv[i] <= 0:
            return heights_f[i]
    return None

transition_points_auto = {}
for col in trpm_columns:
    clean_data = df_new[['h_mm', col]].dropna()
    transition_points_auto[col] = find_transition_point(clean_data['h_mm'], clean_data[col])

print("Transition points computed.")

In [ ]:
# Generate plot and save as SVG
polynomial_fits_normalized = {}
fit_results_normalized = {}

n_samples = len(trpm_columns)
colors = sns.color_palette("tab20", n_colors=n_samples)

fig = plt.figure(figsize=(6, 6))

for i, col in enumerate(trpm_columns):
    clean_data = df_new[['h_mm', col]].dropna()
    trans_height = transition_points_auto.get(col, None)

    if trans_height is not None:
        mask = clean_data['h_mm'] >= trans_height
        filtered_data = clean_data[mask]

        if len(filtered_data) >= 3:
            heights_normalized = filtered_data['h_mm'].values - trans_height
            trpm_values = filtered_data[col].values

            viscosity_label_str = col.replace('_t/rpm', '')
            label_viscosity = extract_viscosity(col)

            plt.scatter(heights_normalized, trpm_values, color=colors[i],
                        label=f'{int(label_viscosity):,} cP', alpha=0.8, s=30)

plt.xlabel('Gap Height (mm)', fontsize=20)
plt.ylabel('Rotational Drag (% of dyne.cm/RPM)', fontsize=20)
plt.legend(fontsize=10, ncol=2, loc='upper right', frameon=True)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.grid(False)

for side in ['top', 'bottom', 'left', 'right']:
    plt.gca().spines[side].set_linewidth(1.5)
    plt.gca().spines[side].set_color('black')

plt.tight_layout()
plt.savefig('normalized_rotational_drag.svg', format='svg', bbox_inches='tight')
plt.show()
print("SVG saved as 'normalized_rotational_drag.svg'")